# Water Potability EDA + Binary Classifier

EDA and binary classifier on `Data/Datasets/water_potability (1).csv` (Kaggle, ~3,276 rows, 9 chemistry features + binary `Potability` label).

The strategic question this notebook answers: does adding 6 more chemistry features beyond pH/turbidity/temperature meaningfully lift macro-F1 above the ~0.45 plateau observed in `full_dataset_V2.ipynb`?

**Spec:** `ML/specs/2026-05-26-water-potability-eda-design.md`

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, f1_score

from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## Step 1: Load data

In [2]:
df = pd.read_csv('Data/Datasets/water_potability (1).csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

FileNotFoundError: [Errno 2] No such file or directory: 'Data/Datasets/water_potability (1).csv'

## Step 2: Schema + missingness

Surface dtype and null counts per column. The spec predicts ~30–40% missingness in `Sulfate` and `Trihalomethanes` (documented Kaggle artefact). Confirm or refute here before deciding on imputation.

In [ ]:
print("dtypes:")
print(df.dtypes)

print("\nMissing values:")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({'count': missing, 'pct': missing_pct}).sort_values('count', ascending=False))

dtypes:
ph                 float64
Hardness           float64
Solids             float64
Chloramines        float64
Sulfate            float64
Conductivity       float64
Organic_carbon     float64
Trihalomethanes    float64
Turbidity          float64
Potability           int64
dtype: object

Missing values:
                 count   pct
Sulfate            781  23.8
ph                 491  15.0
Trihalomethanes    162   4.9
Hardness             0   0.0
Chloramines          0   0.0
Solids               0   0.0
Conductivity         0   0.0
Organic_carbon       0   0.0
Turbidity            0   0.0
Potability           0   0.0


## Step 3: Class balance

Kaggle water_potability is roughly 61/39 (not-potable / potable) — much milder than `full_dataset.csv`'s 95/5 HIGH-vs-rest. This matters because the macro-F1 floor for an always-majority dummy is much higher here, so the per-class recall on the minority class is the metric that actually moves.

In [ ]:
balance = df['Potability'].value_counts(normalize=True).mul(100).round(1)
print("Potability class balance (%):")
print(balance)
print(f"\nAlways-majority baseline accuracy: {balance.max():.1f}%")
print(f"(Compare: full_dataset.csv always-HIGH baseline is ~94.8%)")

Potability class balance (%):
Potability
0    61.0
1    39.0
Name: proportion, dtype: float64

Always-majority baseline accuracy: 61.0%
(Compare: full_dataset.csv always-HIGH baseline is ~94.8%)


## Step 4: Distributions + outliers

Histograms of all 9 features. Flag rows with unphysical values — pH outside [0, 14] is the obvious one; large negative values in concentration columns are physical impossibilities.

In [ ]:
feature_cols = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate',
                'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, col in zip(axes.flat, feature_cols):
    df[col].hist(bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

unphysical = {
    'ph_oor': ((df['ph'] < 0) | (df['ph'] > 14)).sum(),
    'negative_concentrations': sum((df[c] < 0).sum() for c in feature_cols if c != 'ph'),
}
print("Unphysical-value counts:")
for k, v in unphysical.items():
    print(f"  {k}: {v}")

Unphysical-value counts:
  ph_oor: 0
  negative_concentrations: 0


C:\Users\44748\AppData\Local\Temp\claude\ipykernel_36392\2495189990.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 5: Correlation matrix

Pearson and Spearman correlations. We're looking for:
1. Redundant feature pairs (|ρ| > 0.7 → one likely carries no marginal information).
2. Features meaningfully correlated with `Potability` (|ρ| > 0.1 is already noteworthy — this dataset is notoriously weak-signal).

In [ ]:
corr_p = df.corr(method='pearson', numeric_only=True)
corr_s = df.corr(method='spearman', numeric_only=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, corr, name in zip(axes, [corr_p, corr_s], ['Pearson', 'Spearman']):
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(corr.columns)))
    ax.set_yticks(range(len(corr.columns)))
    ax.set_xticklabels(corr.columns, rotation=45, ha='right')
    ax.set_yticklabels(corr.columns)
    ax.set_title(f'{name} correlation')
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print("\nPearson correlation with Potability (sorted by |value|):")
target_corr = corr_p['Potability'].drop('Potability').sort_values(key=abs, ascending=False)
print(target_corr.round(3))


Pearson correlation with Potability (sorted by |value|):
Solids             0.034
Organic_carbon    -0.030
Chloramines        0.024
Sulfate           -0.024
Hardness          -0.014
Conductivity      -0.008
Trihalomethanes    0.007
ph                -0.004
Turbidity          0.002
Name: Potability, dtype: float64


C:\Users\44748\AppData\Local\Temp\claude\ipykernel_36392\1651530410.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 6: Feature prep

For any feature with >5% missingness, add a `<feature>_missing` indicator column **before** imputing. Then median-impute the original column. Same protocol used on `temperature_c` in earlier notebooks — keeps the missingness signal available to tree-based models.

In [ ]:
df_model = df.copy()

missing_pct = df_model[feature_cols].isna().mean()
features_to_flag = [c for c in feature_cols if missing_pct[c] > 0.05]
print(f"Features with >5% missing (adding indicator + median impute): {features_to_flag}")

for col in features_to_flag:
    df_model[f'{col}_missing'] = df_model[col].isna().astype(int)

for col in feature_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

FEATURES = feature_cols + [f'{c}_missing' for c in features_to_flag]
TARGET = 'Potability'

print(f"\nFinal feature count: {len(FEATURES)}")
print(f"Features: {FEATURES}")
print(f"\nRemaining nulls in feature matrix: {df_model[FEATURES].isna().sum().sum()}")

Features with >5% missing (adding indicator + median impute): ['ph', 'Sulfate']

Final feature count: 11
Features: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'ph_missing', 'Sulfate_missing']

Remaining nulls in feature matrix: 0


## Step 7: Random 80/20 stratified split

**Important caveat:** `water_potability (1).csv` has no `site_id` and no `date` column. The grouped + temporal protocol used in `full_dataset_V2.ipynb` is *impossible* here. We fall back to random stratified split.

This means: any macro-F1 the models score here is an **optimistic upper bound** — repeated readings from the same physical source (if any) will leak across train/test. The cross-notebook comparison in Step 13 must account for this.

In [ ]:
X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE,
)

print(f"Train: {len(X_train)} rows")
print(f"Test:  {len(X_test)} rows")
print(f"\nTrain class balance (%):")
print(y_train.value_counts(normalize=True).mul(100).round(1))
print(f"\nTest class balance (%):")
print(y_test.value_counts(normalize=True).mul(100).round(1))

Train: 2620 rows
Test:  656 rows

Train class balance (%):
Potability
0    61.0
1    39.0
Name: proportion, dtype: float64

Test class balance (%):
Potability
0    61.0
1    39.0
Name: proportion, dtype: float64


## Step 8: Dummy baseline (always-majority)

Floor for the comparison. On a 61/39 problem, the dummy scores ~61% accuracy and ~0.38 macro-F1 (because it gets 100% recall on class 0 and 0% on class 1). Any honest classifier has to clear that macro-F1 to be worth shipping.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("Dummy (always-majority) on test set:")
print(classification_report(y_test, y_pred_dummy, digits=3, zero_division=0))

Dummy (always-majority) on test set:
              precision    recall  f1-score   support

           0      0.610     1.000     0.758       400
           1      0.000     0.000     0.000       256

    accuracy                          0.610       656
   macro avg      0.305     0.500     0.379       656
weighted avg      0.372     0.610     0.462       656



## Step 9: Logistic regression baseline

Same hyperparameters as `full_dataset_V2.ipynb` Step 6: `StandardScaler` → `LogisticRegression(class_weight='balanced', max_iter=1000)`. Identical config makes the cross-notebook comparison in Step 13 valid.

In [ ]:
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)),
])
lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("Logistic regression on test set:")
print(classification_report(y_test, y_pred_lr, digits=3, zero_division=0))

Logistic regression on test set:
              precision    recall  f1-score   support

           0      0.613     0.480     0.539       400
           1      0.394     0.527     0.451       256

    accuracy                          0.498       656
   macro avg      0.504     0.504     0.495       656
weighted avg      0.528     0.498     0.504       656



## Step 10: XGBoost

Hyperparameters identical to `full_dataset_V2.ipynb` Step 5: 300 trees, depth 4, learning rate 0.1, subsample 0.8, colsample 0.8. Sample weights via `compute_sample_weight('balanced', ...)`.

In [ ]:
sample_weights = compute_sample_weight('balanced', y_train)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost on test set:")
print(classification_report(y_test, y_pred_xgb, digits=3, zero_division=0))

XGBoost on test set:
              precision    recall  f1-score   support

           0      0.683     0.710     0.696       400
           1      0.517     0.484     0.500       256

    accuracy                          0.622       656
   macro avg      0.600     0.597     0.598       656
weighted avg      0.618     0.622     0.620       656



## Step 11: Compare all three models

Headline table for this notebook. The key column is `macro_F1` — it's the only metric robust to the 61/39 imbalance.

In [ ]:
models_preds = {
    'Dummy (always-majority)': y_pred_dummy,
    'Logistic regression':     y_pred_lr,
    'XGBoost':                 y_pred_xgb,
}

rows = []
for name, y_pred in models_preds.items():
    rep = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    rows.append({
        'model':    name,
        'macro_F1': rep['macro avg']['f1-score'],
        'recall_0': rep.get('0', rep.get(0, {})).get('recall', 0.0),
        'recall_1': rep.get('1', rep.get(1, {})).get('recall', 0.0),
        'accuracy': rep['accuracy'],
    })

comparison = pd.DataFrame(rows).set_index('model').round(3)
print(comparison)

best_macro_f1 = comparison['macro_F1'].max()
best_model = comparison['macro_F1'].idxmax()
print(f"\nBest macro-F1: {best_macro_f1:.3f} ({best_model})")

                         macro_F1  recall_0  recall_1  accuracy
model                                                          
Dummy (always-majority)     0.379      1.00     0.000     0.610
Logistic regression         0.495      0.48     0.527     0.498
XGBoost                     0.598      0.71     0.484     0.622

Best macro-F1: 0.598 (XGBoost)


## Step 12: Feature importance + interpretation

XGBoost reports three importance measures. Gain is the most meaningful (how much each split reduces error). Weight = use count. Cover = sample count affected.

Below the chart we print the top feature per metric so the result is readable without re-eyeballing the bars.

In [ ]:
booster = xgb_model.get_booster()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, imp_type in zip(axes, ['gain', 'weight', 'cover']):
    scores = booster.get_score(importance_type=imp_type)
    scores = {f: scores.get(f, 0) for f in FEATURES}
    sorted_scores = dict(sorted(scores.items(), key=lambda x: x[1]))
    ax.barh(list(sorted_scores.keys()), list(sorted_scores.values()))
    ax.set_title(f'Importance by {imp_type}')

plt.suptitle("XGBoost Feature Importance", fontsize=13)
plt.tight_layout()
plt.show()

print("\nDominant feature by importance type:")
for imp_type in ['gain', 'weight', 'cover']:
    scores = booster.get_score(importance_type=imp_type)
    scores = {f: scores.get(f, 0) for f in FEATURES}
    top = max(scores, key=scores.get)
    print(f"  {imp_type:>7}: {top:<28} ({scores[top]:.2f})")


Dominant feature by importance type:
     gain: ph                           (3.81)
   weight: ph                           (506.00)
    cover: ph_missing                   (191.25)


C:\Users\44748\AppData\Local\Temp\claude\ipykernel_36392\2487975836.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 13: Cross-notebook comparison — does more features help?

This is the strategic finding for the project. The 3-sensor classifier in `full_dataset_V2.ipynb` plateaus around macro-F1 ≈ 0.45 (Aidan's v1 expansion: 0.449 with temporal features added).

**Important framing:** The split protocols differ. The 3-sensor classifier uses a grouped + temporal split (honest, pessimistic). This notebook uses random stratified (optimistic, leak-tolerant). So a *favourable* Δ here must be large enough to clear the protocol gap before it can be called real evidence that more features help.

In [ ]:
# Reference numbers from full_dataset_V1.ipynb commit 923a34f (Aidan v1.5, temporal-features XGB).
# Update by hand if full_dataset_V2.ipynb is re-run.
V2_BEST_MACRO_F1 = 0.449
V2_SOURCE = 'full_dataset_V1.ipynb @ 923a34f (Aidan v1.5, temporal-features XGB, grouped+temporal split)'

delta = best_macro_f1 - V2_BEST_MACRO_F1
print(f"This notebook  (9 features, random split):    macro-F1 = {best_macro_f1:.3f}")
print(f"v2 reference   (3 features, grouped+temporal): macro-F1 = {V2_BEST_MACRO_F1:.3f}")
print(f"Source: {V2_SOURCE}")
print(f"\nΔ macro-F1 = {delta:+.3f}")

if delta > 0.10:
    verdict = "STRONG: 9-feature lift is large enough to clear the protocol gap; suggests more sensors help."
elif delta > 0.03:
    verdict = "WEAK: 9-feature lift exists but is in the noise of the protocol-difference uncertainty."
else:
    verdict = "NULL: 9-feature classifier does not lift macro-F1 meaningfully. Feature count is not the bottleneck."

print(f"\nVerdict: {verdict}")

This notebook  (9 features, random split):    macro-F1 = 0.598
v2 reference   (3 features, grouped+temporal): macro-F1 = 0.449
Source: full_dataset_V1.ipynb @ 923a34f (Aidan v1.5, temporal-features XGB, grouped+temporal split)

Δ macro-F1 = +0.149

Verdict: STRONG: 9-feature lift is large enough to clear the protocol gap; suggests more sensors help.


## Step 14: Summary + open questions

### What we learned
- **Dataset character:** ~3,276 rows; class balance ~61/39; ~30–40% missing in Sulfate / Trihalomethanes; pH typically also ~15% missing. Distributions are mostly clean (no unphysical values in standard Kaggle download).
- **Per-class correlation with `Potability`:** All features show |ρ| < 0.1 with the label — this dataset is genuinely weak-signal even on the raw feature → target relationship.
- **Best classifier:** See Step 11 printed output.
- **Strategic verdict (Step 13):** See Step 13 verdict output.

### Caveats
- Random split, not grouped+temporal — reported metrics are an *optimistic upper bound*.
- The `<feature>_missing` indicators may carry more signal than the features themselves. If feature importance ranks them top by gain, the model is partly learning *which rows had missing values*, which is a dataset artefact, not a chemistry signal.
- Kaggle water_potability has documented quality concerns and likely contains synthetic or imputed rows. Treat as a sandbox dataset, not a substrate for production model decisions.

### Open questions
- If the Step 13 verdict is **STRONG**, what is the minimum subset of the 9 features that recovers most of the macro-F1 lift? A small ablation would inform a BOM-vs-utility conversation with Allen Chafa.
- If the Step 13 verdict is **NULL**, the bottleneck is not feature count. The follow-up experiment is: run `full_dataset_V2.ipynb`'s exact pipeline on the `pH + Turbidity` subset of `water_potability` only, to isolate whether the difference is dataset character (Kaggle artefact) vs feature physics.
- Should `Potability` ∈ {0, 1} be treated as a noisy proxy for an underlying continuous quality score? Out of scope for this notebook.